# LoRA fine-tune — Qwen3-VL-2B Persian line OCR (Colab T4)

Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Everything persistent goes to Google Drive under `MyDrive/exam-ocr/`, so a disconnect resumes from the last checkpoint when you re-run the training cell.

Step-by-step: `training/lora/README.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/exam-ocr'
import os; os.makedirs(DRIVE, exist_ok=True)

## Code
If the repository is private, replace the URL with `https://<TOKEN>@github.com/alireza-taheriF/exam-ocr.git` (a fine-grained read-only token).

In [ ]:
BRANCH = 'claude/plan-90-feedback-loop'
!rm -rf /content/exam-ocr && git clone -q -b $BRANCH https://github.com/alireza-taheriF/exam-ocr.git /content/exam-ocr
%cd /content/exam-ocr
!git log --oneline -1

In [ ]:
!pip -q install 'transformers>=5.0' peft accelerate datasets python-dotenv

## Local data
Upload `exam_ocr_lora_data_v3.zip` (883 Claude-labeled exam lines + 4000 MR3z4 handwriting lines; from `training/lora/make_bundle.py`) to `MyDrive/exam-ocr/`. It carries the handwriting line crops built by `scripts/build_hw_lines.py` (MR3z4 train + validation, ~28k lines) and, once students have corrected sheets, their exported corrections. **Required**: MR3z4 has no image column on the Hub, so it can only come in through this bundle.

In [ ]:
import glob, os
BUNDLE = f'{DRIVE}/exam_ocr_lora_data_v3.zip'
LOCAL = []
if os.path.exists(BUNDLE):
    !rm -rf /content/data && mkdir -p /content/data && unzip -q -o $BUNDLE -d /content/data
    !cat /content/data/bundle_summary.json
    LOCAL = sorted(glob.glob('/content/data/manifests/*.jsonl'))
print('local manifests:', LOCAL)
assert LOCAL, 'upload exam_ocr_lora_data_v3.zip to MyDrive/exam-ocr/ first'

## Train
`NAME` is the adapter version. Defaults fit a free T4 (~15 GB): batch 2 × grad-accum 8, fp16 AMP with fp32 LoRA weights, gradient checkpointing, image long side ≤ 768 px. If you hit CUDA OOM, set `--batch-size 1 --grad-accum 16` or `--max-long-side 512`.

`--max-samples 16000` keeps one epoch near 1000 optimizer steps so it fits a free session; checkpoints go to Drive every 200 steps, so a disconnect only costs the last few minutes — re-run this cell to resume. Replay rows come from `Persian_Arabic_TextLine_Image_Ocr_Small` (columns `Image`/`Text`, split `Train`, matched automatically).

In [ ]:
NAME = 'lora_v3'
OUT = f'{DRIVE}/{NAME}'
import os, glob
if not os.path.exists('/content/exam-ocr/training/lora/train_qwen_lora.py'):  # clone cell was skipped or failed
    !rm -rf /content/exam-ocr && git clone -b claude/plan-90-feedback-loop https://github.com/alireza-taheriF/exam-ocr.git /content/exam-ocr
%cd /content/exam-ocr
assert os.path.exists('training/lora/train_qwen_lora.py'), 'repo clone failed; see the error above'
LOCAL = sorted(glob.glob('/content/data/manifests/*.jsonl'))
assert LOCAL, 'no manifests in /content/data: run the Local data cell first'
local_args = ' '.join(LOCAL)
!python training/lora/train_qwen_lora.py --out $OUT --device cuda \
    --local {local_args} --replay-ratio 0.3 --max-samples 16000 --data-dir /content/ocr_data \
    --epochs 3 --batch-size 2 --grad-accum 8 --lr 5e-5 --lora-r 16


In [ ]:
import json
meta = json.load(open(f'{OUT}/training_meta.json'))
print({k: meta[k] for k in ['n_train','train_counts','steps','loss_first','loss_last','val_cer_before','val_cer_after','minutes']})

## Package the adapter for download
Download `MyDrive/exam-ocr/<NAME>_adapter.zip` and unzip it into `models/<NAME>/` on the Mac.

In [ ]:
!cd $OUT/.. && rm -f {NAME}_adapter.zip && zip -q -r {NAME}_adapter.zip {NAME} -x '{NAME}/checkpoint-*/*' && ls -lh {NAME}_adapter.zip

## Then, on the Mac (not here)
```bash
unzip ~/Downloads/lora_v3_adapter.zip -d models/
HF_HUB_OFFLINE=1 .venv-qwen/bin/python scripts/eval_gate.py --label lora_v3 --adapter models/lora_v3 --register
```
Activate with `QWEN_ADAPTER_PATH=models/lora_v3` **only** if the registry says `champion`.